# 20 — Plan B: User Scenario / Recommendation Engine

Notebook 20 is the operational Plan B scenario engine.

User inputs:
- latitude
- longitude
- target/minimum length
- target/minimum width
- optional DTV
- optional Baustoff

The engine:
1. builds a geographic reference cohort;
2. calculates similarity;
3. adds condition/performance evidence;
4. aggregates evidence by Bauwerksart;
5. produces a transparent Bauwerksart recommendation;
6. returns the reference bridges supporting the result.

The frozen 86-predictor condition model is not retrained, refit, tuned, or modified.

This is a planning decision-support engine. It is not FEM, structural dimensioning, code checking, or formal approval.

## 01 — Current project inputs

Current roots only:

```text
C:\Datenanalyse\final Project\Dataset_PlanA-B
C:\Datenanalyse\final Project\Output_PlanA-B
```

Input:

```text
Output_PlanA-B/
└── 18_Plan_B_Reference_Library/
    └── plan_b_reference_library.parquet
```

Outputs:

```text
Output_PlanA-B/
└── 20_Plan_B_User_Scenario/
    ├── plan_b_scenario_result.csv
    ├── plan_b_scenario_references.csv
    ├── plan_b_scenario_type_summary.csv
    └── 20_plan_b_scenario_manifest.json
```

In [1]:
from pathlib import Path
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Datenanalyse\final Project")
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

LIBRARY_INPUT = (
    OUTPUT_ROOT
    / "18_Plan_B_Reference_Library"
    / "plan_b_reference_library.parquet"
)

OUTPUT_DIR = OUTPUT_ROOT / "20_Plan_B_User_Scenario"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_CSV = OUTPUT_DIR / "plan_b_scenario_result.csv"
REFERENCES_CSV = OUTPUT_DIR / "plan_b_scenario_references.csv"
TYPE_SUMMARY_CSV = OUTPUT_DIR / "plan_b_scenario_type_summary.csv"
MANIFEST_JSON = OUTPUT_DIR / "20_plan_b_scenario_manifest.json"

if not LIBRARY_INPUT.exists():
    raise FileNotFoundError(
        f"Notebook 18 reference library not found:\n{LIBRARY_INPUT}"
    )

reference = pd.read_parquet(LIBRARY_INPUT)

if len(reference) != 52214:
    raise ValueError(
        f"Expected 52,214 reference bridges; found {len(reference)}"
    )

if not reference["bridge_id"].is_unique:
    raise ValueError("Reference library bridge_id must be unique.")

print("[PASS] Notebook 18 reference library loaded")
print("Rows:", len(reference))

[PASS] Notebook 18 reference library loaded
Rows: 52214


## 02 — User scenario

Required:
- latitude
- longitude
- length_m
- width_m

Optional:
- dtv
- baustoff

Length and width are Plan B similarity inputs. They are not inserted into the frozen 86-predictor condition model.

Missing DTV is not interpreted as zero.

In [2]:
# ============================================================
# USER SCENARIO — EDIT THESE VALUES
# ============================================================

SCENARIO = {
    "latitude": 50.7374,
    "longitude": 7.0982,
    "length_m": 80.0,
    "width_m": 12.0,
    "dtv": None,
    "baustoff": None,
}

TOP_REFERENCES_PER_TYPE = 10
MIN_REFERENCES_PER_TYPE = 5
MAX_DISTANCE_KM = 50.0

for field in ["latitude", "longitude", "length_m", "width_m"]:
    if SCENARIO[field] is None:
        raise ValueError(f"Required scenario field is missing: {field}")

SCENARIO["latitude"] = float(SCENARIO["latitude"])
SCENARIO["longitude"] = float(SCENARIO["longitude"])
SCENARIO["length_m"] = float(SCENARIO["length_m"])
SCENARIO["width_m"] = float(SCENARIO["width_m"])

if not -90 <= SCENARIO["latitude"] <= 90:
    raise ValueError("latitude must be between -90 and 90.")
if not -180 <= SCENARIO["longitude"] <= 180:
    raise ValueError("longitude must be between -180 and 180.")
if SCENARIO["length_m"] <= 0 or SCENARIO["width_m"] <= 0:
    raise ValueError("length_m and width_m must be > 0.")

if SCENARIO["dtv"] is not None:
    SCENARIO["dtv"] = float(SCENARIO["dtv"])
    if SCENARIO["dtv"] < 0:
        raise ValueError("dtv must be >= 0.")

if SCENARIO["baustoff"] is not None:
    SCENARIO["baustoff"] = str(SCENARIO["baustoff"]).strip()

print("SCENARIO")
for key, value in SCENARIO.items():
    print(f"  {key}: {value}")

print("[PASS] scenario input gate")

SCENARIO
  latitude: 50.7374
  longitude: 7.0982
  length_m: 80.0
  width_m: 12.0
  dtv: None
  baustoff: None
[PASS] scenario input gate


## 03 — Decision configuration

Agreed initial weights:

```text
Bauwerksart   30%
Baustoff      20%
Länge         15%
Breite        10%
DTV           15%
Distanz       10%
```

For a new-bridge recommendation, Bauwerksart is the target and cannot be an input. Its 30% is excluded.

If DTV or Baustoff is not supplied, that component is excluded.

The remaining active weights are renormalized.

In [3]:
BASE_WEIGHTS = {
    "bauwerksart": 0.30,
    "baustoff": 0.20,
    "length": 0.15,
    "width": 0.10,
    "dtv": 0.15,
    "distance": 0.10,
}

active = {
    "baustoff": SCENARIO["baustoff"] is not None,
    "length": True,
    "width": True,
    "dtv": SCENARIO["dtv"] is not None,
    "distance": True,
}

raw_weights = {
    name: BASE_WEIGHTS[name]
    for name, enabled in active.items()
    if enabled
}

weight_sum = sum(raw_weights.values())
ACTIVE_WEIGHTS = {name: value / weight_sum for name, value in raw_weights.items()}

print("ACTIVE_WEIGHTS")
for name, value in ACTIVE_WEIGHTS.items():
    print(f"  {name}: {value:.6f}")

print("[PASS] decision configuration")

ACTIVE_WEIGHTS
  length: 0.428571
  width: 0.285714
  distance: 0.285714
[PASS] decision configuration


In [4]:
def haversine_km(lat, lon, ref_lat, ref_lon):
    lat1 = np.radians(float(lat))
    lon1 = np.radians(float(lon))
    lat2 = np.radians(np.asarray(ref_lat, dtype=float))
    lon2 = np.radians(np.asarray(ref_lon, dtype=float))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    return 6371.0088 * 2.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def numeric_similarity(series, value, scale):
    x = pd.to_numeric(series, errors="coerce")
    return pd.Series(
        np.where(
            x.notna(),
            np.exp(-np.abs(x - float(value)) / float(scale)),
            np.nan,
        ),
        index=series.index,
    )


def categorical_similarity(series, value):
    if value is None:
        return pd.Series(np.nan, index=series.index)
    return series.astype("string").eq(str(value)).astype(float)


def dtv_similarity(series, value):
    if value is None:
        return pd.Series(np.nan, index=series.index)

    x = pd.to_numeric(series, errors="coerce")
    v = float(value)
    denominator = np.maximum(np.maximum(np.abs(x), abs(v)), 1.0)
    score = np.clip(1.0 - np.abs(x - v) / denominator, 0.0, 1.0)

    return pd.Series(
        np.where(x.notna(), score, np.nan),
        index=series.index,
    )


length_scale = max(float(reference["laenge"].dropna().quantile(0.75)), 1.0)
width_scale = max(float(reference["breite"].dropna().quantile(0.75)), 1.0)

print("[PASS] similarity functions")

[PASS] similarity functions


## 04 — Geographic reference cohort

Only existing bridges within the configured radius are used for the main scenario cohort.

Default radius:

```text
50 km
```

Distance remains an independent similarity component.

In [5]:
work = reference.copy()

work["distance_km"] = haversine_km(
    SCENARIO["latitude"],
    SCENARIO["longitude"],
    work["latitude"].to_numpy(),
    work["longitude"].to_numpy(),
)

work = work[
    work["distance_km"].notna()
    & (work["distance_km"] <= MAX_DISTANCE_KM)
].copy()

if work.empty:
    raise ValueError(
        f"No reference bridges found within {MAX_DISTANCE_KM} km."
    )

print("Reference bridges within radius:", len(work))
print("Minimum distance:", round(work["distance_km"].min(), 3), "km")
print("Maximum distance:", round(work["distance_km"].max(), 3), "km")
print("[PASS] geographic cohort")

Reference bridges within radius: 2412
Minimum distance: 1.62 km
Maximum distance: 49.988 km
[PASS] geographic cohort


In [6]:
if SCENARIO["baustoff"] is not None:
    work["sim_baustoff"] = categorical_similarity(
        work["baustoffklasse"],
        SCENARIO["baustoff"],
    )

work["sim_length"] = numeric_similarity(
    work["laenge"],
    SCENARIO["length_m"],
    length_scale,
)

work["sim_width"] = numeric_similarity(
    work["breite"],
    SCENARIO["width_m"],
    width_scale,
)

if SCENARIO["dtv"] is not None:
    work["sim_dtv"] = dtv_similarity(
        work["dtv_reference"],
        SCENARIO["dtv"],
    )

work["sim_distance"] = np.exp(-work["distance_km"] / 50.0)

component_columns = {
    "baustoff": "sim_baustoff",
    "length": "sim_length",
    "width": "sim_width",
    "dtv": "sim_dtv",
    "distance": "sim_distance",
}

numerator = np.zeros(len(work))
denominator = np.zeros(len(work))

for component, column in component_columns.items():
    if component not in ACTIVE_WEIGHTS:
        continue

    available = work[column].notna().to_numpy()
    weight = ACTIVE_WEIGHTS[component]

    numerator += np.where(
        available,
        work[column].fillna(0).to_numpy() * weight,
        0.0,
    )
    denominator += np.where(available, weight, 0.0)

work["similarity_score"] = np.where(
    denominator > 0,
    numerator / denominator,
    np.nan,
)

work = work[work["similarity_score"].notna()].copy()

if work.empty:
    raise ValueError("No reference bridge can be scored.")

print("[PASS] bridge-level similarity")

[PASS] bridge-level similarity


## 05 — Performance evidence

The reference library contains current and future condition evidence.

For the base Bauwerksart recommendation, current performance evidence is used.

```text
performance_score
```

This is secondary evidence about reference-bridge performance, not a substitute for structural design.

In [7]:
work["performance_score"] = pd.to_numeric(
    work["performance_score"],
    errors="coerce",
)

work["decision_score"] = (
    0.80 * work["similarity_score"]
    + 0.20 * work["performance_score"].fillna(0.0)
)

print(
    "Condition evidence coverage:",
    int(work["performance_score"].notna().sum()),
    "/",
    len(work),
)
print("[PASS] performance evidence")

Condition evidence coverage: 2412 / 2412
[PASS] performance evidence


## 06 — Bauwerksart recommendation

The engine does not copy the type of one nearest bridge.

For each Bauwerksart with enough references:
1. sort references by bridge-level decision score;
2. retain the strongest reference cohort;
3. calculate mean similarity and performance;
4. calculate the combined type score.

This produces a cohort-based recommendation.

In [8]:
work = work.sort_values(
    ["decision_score", "similarity_score"],
    ascending=[False, False],
)

type_rows = []

for bauwerksart, group in work.groupby(
    "bauwerksart_text",
    dropna=False,
):
    group = group.sort_values(
        ["decision_score", "similarity_score"],
        ascending=[False, False],
    )

    if len(group) < MIN_REFERENCES_PER_TYPE:
        continue

    cohort = group.head(TOP_REFERENCES_PER_TYPE)

    type_rows.append({
        "bauwerksart": bauwerksart,
        "reference_count_within_radius": len(group),
        "cohort_size_used": len(cohort),
        "mean_similarity": float(cohort["similarity_score"].mean()),
        "mean_performance": float(cohort["performance_score"].mean()),
        "mean_distance_km": float(cohort["distance_km"].mean()),
        "mean_length_m": float(cohort["laenge"].mean()),
        "mean_width_m": float(cohort["breite"].mean()),
        "mean_dtv": float(cohort["dtv_reference"].mean()),
        "type_decision_score": float(
            0.80 * cohort["similarity_score"].mean()
            + 0.20 * cohort["performance_score"].fillna(0.0).mean()
        ),
    })

type_summary = pd.DataFrame(type_rows)

if type_summary.empty:
    raise ValueError(
        "No Bauwerksart has enough comparable references within the geographic cohort."
    )

type_summary = type_summary.sort_values(
    ["type_decision_score", "mean_similarity"],
    ascending=[False, False],
).reset_index(drop=True)

type_summary["decision_rank"] = np.arange(1, len(type_summary) + 1)

recommended_type = type_summary.iloc[0]["bauwerksart"]

print("Recommended Bauwerksart:", recommended_type)
print(type_summary.head(15).to_string(index=False))
print("[PASS] Bauwerksart recommendation")

Recommended Bauwerksart: Plattenbalkenbrücke, Trägerrostbrücke
                                     bauwerksart  reference_count_within_radius  cohort_size_used  mean_similarity  mean_performance  mean_distance_km  mean_length_m  mean_width_m      mean_dtv  type_decision_score  decision_rank
           Plattenbalkenbrücke, Trägerrostbrücke                            613                10         0.851170          0.636667         20.387121      81.797000     13.191000  38161.333333             0.808270              1
                                   Plattenbrücke                            845                10         0.808839          0.610000         25.653809      78.156000     10.066000  58939.000000             0.769071              2
                                Hohlkastenbrücke                            154                10         0.812409          0.530000         10.970194      80.031000     10.394000  80121.750000             0.755927              3
      Balkenbrück

## 07 — Supporting reference bridges

The strongest reference bridges belonging to the recommended Bauwerksart are returned as evidence.

The final result therefore contains both:
- the recommendation;
- the reference bridges supporting it.

In [9]:
recommended_refs = (
    work[
        work["bauwerksart_text"].astype("string")
        == str(recommended_type)
    ]
    .sort_values(
        ["decision_score", "similarity_score"],
        ascending=[False, False],
    )
    .head(TOP_REFERENCES_PER_TYPE)
    .copy()
)

if len(recommended_refs) < MIN_REFERENCES_PER_TYPE:
    raise RuntimeError("Insufficient evidence references.")

reference_columns = [
    "bridge_id",
    "bauwerksart_text",
    "baustoffklasse",
    "latitude",
    "longitude",
    "distance_km",
    "laenge",
    "breite",
    "dtv_reference",
    "zustandsnote_observed",
    "zustandsnote_predicted",
    "zustandsnote_tplus_10y",
    "zustandsnote_tplus_25y",
    "zustandsnote_tplus_50y",
    "similarity_score",
    "performance_score",
    "decision_score",
]

recommended_refs = recommended_refs[
    [c for c in reference_columns if c in recommended_refs.columns]
]

print("Evidence references:", len(recommended_refs))
print("[PASS] recommendation evidence")

Evidence references: 10
[PASS] recommendation evidence


## 08 — Export scenario result

The result records:
- user inputs;
- recommended Bauwerksart;
- type-level score;
- similarity evidence;
- performance evidence;
- geographic cohort size;
- reference cohort size;
- active weights.

In [10]:
recommended_row = type_summary.iloc[0]

result = {
    "scenario_latitude": SCENARIO["latitude"],
    "scenario_longitude": SCENARIO["longitude"],
    "scenario_length_m": SCENARIO["length_m"],
    "scenario_width_m": SCENARIO["width_m"],
    "scenario_dtv": SCENARIO["dtv"],
    "scenario_baustoff": SCENARIO["baustoff"],
    "recommended_bauwerksart": recommended_type,
    "decision_rank": int(recommended_row["decision_rank"]),
    "type_decision_score": float(recommended_row["type_decision_score"]),
    "mean_similarity_top_cohort": float(recommended_row["mean_similarity"]),
    "mean_performance_top_cohort": float(recommended_row["mean_performance"]),
    "reference_count_within_radius": int(
        recommended_row["reference_count_within_radius"]
    ),
    "cohort_size_used": int(recommended_row["cohort_size_used"]),
    "mean_distance_km_top_cohort": float(
        recommended_row["mean_distance_km"]
    ),
    "max_distance_km": MAX_DISTANCE_KM,
    "active_similarity_weights": json.dumps(
        ACTIVE_WEIGHTS,
        ensure_ascii=False,
    ),
    "frozen_model_changed": False,
    "fem_performed": False,
}

result_df = pd.DataFrame([result])

result_df.to_csv(RESULT_CSV, index=False)
recommended_refs.to_csv(REFERENCES_CSV, index=False)
type_summary.to_csv(TYPE_SUMMARY_CSV, index=False)

print(result_df.T.to_string(header=False))
print("[PASS] scenario outputs")

scenario_latitude                                                                                                    50.7374
scenario_longitude                                                                                                    7.0982
scenario_length_m                                                                                                       80.0
scenario_width_m                                                                                                        12.0
scenario_dtv                                                                                                            None
scenario_baustoff                                                                                                       None
recommended_bauwerksart                                                                Plattenbalkenbrücke, Trägerrostbrücke
decision_rank                                                                                                              1


## 09 — Manifest

The scenario, reference-library hash, weights, cohort radius and output paths are recorded.

Changing only the scenario values and rerunning Notebook 20 creates a new scenario result without changing the reference library or frozen model.

In [11]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "notebook": "20_Plan_B_User_Scenario_Recommendation_Engine",
    "timestamp_local": datetime.now().isoformat(timespec="seconds"),
    "reference_library": str(LIBRARY_INPUT),
    "reference_library_sha256": sha256_file(LIBRARY_INPUT),
    "scenario": SCENARIO,
    "base_weights": BASE_WEIGHTS,
    "active_weights": ACTIVE_WEIGHTS,
    "max_distance_km": MAX_DISTANCE_KM,
    "top_references_per_type": TOP_REFERENCES_PER_TYPE,
    "min_references_per_type": MIN_REFERENCES_PER_TYPE,
    "decision_formula": (
        "0.80 * mean_similarity_top_cohort + "
        "0.20 * mean_performance_top_cohort"
    ),
    "frozen_condition_model_changed": False,
    "frozen_condition_model_retrained": False,
    "fem_performed": False,
    "outputs": {
        "scenario_result": str(RESULT_CSV),
        "scenario_references": str(REFERENCES_CSV),
        "type_summary": str(TYPE_SUMMARY_CSV),
    },
}

MANIFEST_JSON.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("[PASS] manifest:", MANIFEST_JSON)

[PASS] manifest: C:\Datenanalyse\final Project\Output_PlanA-B\20_Plan_B_User_Scenario\20_plan_b_scenario_manifest.json


## 10 — Final Notebook 20 gate

The operational scenario engine is complete when:
- scenario inputs are valid;
- a geographic cohort exists;
- similarity scores are finite;
- at least one Bauwerksart has enough references;
- a recommendation exists;
- evidence references exist;
- all outputs are written;
- the frozen model remains unchanged.

In [12]:
finite_similarity = np.isfinite(
    work["similarity_score"].dropna()
).all()

final_checks = {
    "scenario_input_valid": True,
    "geographic_cohort_nonempty": len(work) > 0,
    "finite_similarity": finite_similarity,
    "type_summary_nonempty": len(type_summary) > 0,
    "recommendation_present": pd.notna(recommended_type),
    "evidence_references_present": len(recommended_refs) >= MIN_REFERENCES_PER_TYPE,
    "result_csv_exists": RESULT_CSV.exists(),
    "references_csv_exists": REFERENCES_CSV.exists(),
    "type_summary_exists": TYPE_SUMMARY_CSV.exists(),
    "manifest_exists": MANIFEST_JSON.exists(),
    "frozen_model_unchanged": True,
}

print("FINAL NOTEBOOK 20 GATE")

for name, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(final_checks.values()):
    raise RuntimeError("Notebook 20 final gate failed.")

print()
print("20 STATUS: COMPLETE")
print("Recommended Bauwerksart:", recommended_type)
print("Scenario result:", RESULT_CSV)

FINAL NOTEBOOK 20 GATE
[PASS] scenario_input_valid
[PASS] geographic_cohort_nonempty
[PASS] finite_similarity
[PASS] type_summary_nonempty
[PASS] recommendation_present
[PASS] evidence_references_present
[PASS] result_csv_exists
[PASS] references_csv_exists
[PASS] type_summary_exists
[PASS] manifest_exists
[PASS] frozen_model_unchanged

20 STATUS: COMPLETE
Recommended Bauwerksart: Plattenbalkenbrücke, Trägerrostbrücke
Scenario result: C:\Datenanalyse\final Project\Output_PlanA-B\20_Plan_B_User_Scenario\plan_b_scenario_result.csv
